# **Machine Learning**

## Import dataset

In [1]:
# Import libraries
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
# Loading the data
input_path = Path("..") / 'data' / 'processed' / 'dataset_processed.csv'
df = pd.read_csv(input_path)
display(df.head(5))
print("Shape of the piezometer dataframe:\n", df.shape)

,latitude,longitude,temperature_2m_max,sunrise,sunset,daylight_duration,precipitation_sum,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,...,SPLI,SPI,SETI,SSTI,SSRI,SWSI,SCCI,SPMI,SPEI,SSMI
0,42.685696,2.685378,9.8,2017-01-01T07:19,2017-01-01T16:26,32800.70,0.0,7.88,0.95,47,...,0.124164,0.66519,0.380523,-1.018795,-0.124164,0.66519,-1.018795,0.66519,0.66519,0.124164
1,42.685696,2.685378,9.0,2017-01-02T07:19,2017-01-02T16:27,32850.58,0.0,4.29,0.56,93,...,0.124164,0.66519,0.380523,-1.018795,-0.124164,0.66519,-1.018795,0.66519,0.66519,0.124164
2,42.685696,2.685378,11.7,2017-01-03T07:19,2017-01-03T16:28,32904.28,0.0,8.02,1.59,3,...,0.124164,0.66519,0.380523,-1.018795,-0.124164,0.66519,-1.018795,0.66519,0.66519,0.124164
3,42.685696,2.685378,10.6,2017-01-04T07:19,2017-01-04T16:29,32961.75,0.0,8.08,2.52,5,...,0.124164,0.66519,0.380523,-1.018795,-0.124164,0.66519,-1.018795,0.66519,0.66519,0.124164
4,42.685696,2.685378,7.1,2017-01-05T07:19,2017-01-05T16:30,33022.91,0.0,6.93,2.07,14,...,0.124164,0.66519,0.380523,-1.018795,-0.124164,0.66519,-1.018795,0.66519,0.66519,0.124164


Shape of the piezometer dataframe:
 (3487, 36)


In [3]:
df.columns.tolist()

['latitude',
 'longitude',
 'temperature_2m_max',
 'sunrise',
 'sunset',
 'daylight_duration',
 'precipitation_sum',
 'shortwave_radiation_sum',
 'et0_fao_evapotranspiration',
 'cloud_cover_mean',
 'pressure_msl_mean',
 'wind_speed_10m_mean',
 'soil_moisture_0_to_100cm_mean',
 'soil_temperature_0_to_100cm_mean',
 'code_bss',
 'date_index',
 'bss_id',
 'niveau_nappe_eau',
 'mode_obtention',
 'nom_producteur',
 'P_cum_30d',
 'P_cum_90d',
 'Peff_cum_30d',
 'Peff_cum_90d',
 'Temperature_mean_30d',
 'Temperature_mean_90d',
 'SPLI',
 'SPI',
 'SETI',
 'SSTI',
 'SSRI',
 'SWSI',
 'SCCI',
 'SPMI',
 'SPEI',
 'SSMI']

## Linear regression

In [4]:
# Import libraries
from src.models.preprocessing import preprocessing
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

In [5]:
def linear_regression(df):
    """
    Function to run a Linear Regression on a train and test set.
    Return the prediction and evaluation metrics (R2 on train and test, RMSE, MAE)
    """
    model = LinearRegression()
    X_train, y_train_1m, y_train_2m, y_train_3m, X_test, y_test_1m, y_test_2m, y_test_3m = preprocessing(df)
    
    # Training of the model
    fit_1m = model.fit(X_train, y_train_1m)
    fit_2m = model.fit(X_train, y_train_2m)
    fit_3m = model.fit(X_train, y_train_3m)
    
    # Prediction
    y_pred_1m = fit_1m.predict(X_test)
    y_pred_2m = fit_2m.predict(X_test)
    y_pred_3m = fit_3m.predict(X_test)
    
    # Evaluation metrics
    r2_train_1m = model.score(X_train, y_train_1m)
    r2_test_1m = model.score(X_test, y_test_1m)
    rmse_1m = np.sqrt(mean_squared_error(y_test_1m, y_pred_1m))
    mae_1m = mean_absolute_error(y_test_1m, y_pred_1m)
    
    r2_train_2m = model.score(X_train, y_train_2m)
    r2_test_2m = model.score(X_test, y_test_2m)
    rmse_2m = np.sqrt(mean_squared_error(y_test_2m, y_pred_2m))
    mae_2m = mean_absolute_error(y_test_2m, y_pred_2m)
        
    r2_train_3m = model.score(X_train, y_train_3m)
    r2_test_3m = model.score(X_test, y_test_3m)
    rmse_3m = np.sqrt(mean_squared_error(y_test_3m, y_pred_3m))
    mae_3m = mean_absolute_error(y_test_3m, y_pred_3m)  
    
    predictions = pd.DataFrame({
        'y_test_1m': y_test_1m.values,
        'y_pred_1m': y_pred_1m,
        'y_test_2m': y_test_2m.values,
        'y_pred_2m': y_pred_2m,
        'y_test_3m': y_test_3m.values,
        'y_pred_3m': y_pred_3m}, index=y_test_1m.index)
    
    
    evaluation_metrics_lr = pd.DataFrame({
        'R2_train': [r2_train_1m, r2_train_2m, r2_train_3m],
        'R2_test': [r2_test_1m, r2_test_2m, r2_test_3m],
        'RMSE': [rmse_1m, rmse_2m, rmse_3m],
        'MAE': [mae_1m, mae_2m, mae_3m]}, index = ['Prediction_1m', 'Prediction_2m', 'Prediction_3m'])

    return predictions, evaluation_metrics_lr

In [6]:
predictions, evaluation_metrics_lr = linear_regression(df)

## XGBoost Regressor

In [ ]:
# Import libraries
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
def xgboost_regressor(df, n_estimators=100, learning_rate=0.1, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8, random_state=42):
    """
    Function to run a XGBoost Regressor on a train and test set.
    Return the prediction and evaluation metrics (R2 on train and test, RMSE, MAE)
    """
    X_train, y_train_1m, y_train_2m, y_train_3m, X_test, y_test_1m, y_test_2m, y_test_3m = preprocessing(df)

    y_trains = [y_train_1m, y_train_2m, y_train_3m]
    y_tests  = [y_test_1m,  y_test_2m,  y_test_3m]

    preds   = {}
    metrics = {'R2_train': [], 'R2_test': [], 'RMSE': [], 'MAE': []}

    for i, (y_train, y_test) in enumerate(zip(y_trains, y_tests), start=1):
        model = XGBRegressor(
            n_estimators     = n_estimators,
            learning_rate    = learning_rate,
            max_depth        = max_depth,
            subsample        = subsample,
            colsample_bytree = colsample_bytree,
            random_state     = random_state)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        preds[f'y_test_{i}m'] = y_test.values
        preds[f'y_pred_{i}m'] = y_pred

        metrics['R2_train'].append(model.score(X_train, y_train))
        metrics['R2_test'].append(model.score(X_test, y_test))
        metrics['RMSE'].append(np.sqrt(mean_squared_error(y_test, y_pred)))
        metrics['MAE'].append(mean_absolute_error(y_test, y_pred))

    predictions_xgboost_reg = pd.DataFrame(preds, index=y_test_1m.index)
    evaluation_metrics_xgboost_reg = pd.DataFrame(metrics, index=['Prediction_1m', 'Prediction_2m', 'Prediction_3m'])

    return predictions_xgboost_reg, evaluation_metrics_xgboost_reg

In [ ]:
predictions_xgboost_reg, evaluation_metrics_xgboost_reg = xgboost_regressor(df)